# World Bank Data Preprocessing and Panel Harmonization (2000–2025)

In [1]:
import pandas as pd
from pathlib import Path
import os
import pycountry

In [2]:
#Economics
df_gdp = pd.read_csv("economics/gdp.csv", skiprows=4)
df_inflation = pd.read_csv("economics/inflation.csv", skiprows=4)
df_unemp = pd.read_csv("economics/unemp.csv", skiprows=4)
#Inequality
df_gini = pd.read_csv("inequality/gini.csv", skiprows=4)
df_poverty = pd.read_csv("inequality/poverty.csv", skiprows=4)
#Laborhood
df_labor = pd.read_csv("labor/labor.csv", skiprows=4)
df_literacy = pd.read_csv("education/literacy.csv", skiprows=4)
df_tertiaryeduc = pd.read_csv("education/tertiaryeduc.csv", skiprows=4)
#Governance
df_governance = pd.read_csv("governance/government.csv", skiprows=4) 
df_homicides = pd.read_csv("governance/homicides.csv", skiprows=4) 

In [3]:
import pandas as pd

def clean_wdi_like_df_v2(
    df: pd.DataFrame,
    country_col: str = "Country Name",
    code_col: str = "Country Code",
    min_year: int = 2000,
    value_name: str = "score",
    # choose ONE filtering strategy:
    country_missing_threshold: float | None = 30.0,  # annual indicators
    min_non_missing_years: int | None = None,        # sparse indicators
    # improved missing handling
    interpolate: bool = True,
    interpolate_area: str = "inside",   # "inside" recommended (no edge extrapolation)
    max_gap_years: int | None = 2,      # fill only small internal gaps (annual default = 2)
    # optional edge fill (conservative)
    edge_fill: bool = False,
    edge_fill_limit: int = 1,           # fill at most 1 year at each edge if enabled
    # optional extra coverage rule (recommended for annual panels)
    min_observed_years: int | None = None,  # e.g., 15 for annual (out of ~24 years 2000-2023)
    # missingness flag
    add_missing_flag: bool = True
):
    df = df.copy()

    # 1) Detect year columns
    year_cols_all = [c for c in df.columns if str(c).strip().isdigit()]
    if not year_cols_all:
        raise ValueError("No year columns detected. Expected columns like '1960', '1961', ...")

    # 2) Keep only years >= min_year
    year_cols = [c for c in year_cols_all if int(c) >= min_year]
    if not year_cols:
        raise ValueError(f"No year columns >= {min_year} were found.")
    df = df[[country_col, code_col] + year_cols].copy()

    # 3) Coerce year columns to numeric
    df[year_cols] = df[year_cols].apply(pd.to_numeric, errors="coerce")

    # 4) Drop countries with all years missing (100% missing)
    all_null_mask = df[year_cols].isna().all(axis=1)
    dropped_all_null = df.loc[all_null_mask, [country_col, code_col]].copy()
    df = df.loc[~all_null_mask].copy()

    # 5) Overall missing percentage (only year cells) after year filter
    total_cells = df[year_cols].size
    missing_cells = df[year_cols].isna().sum().sum()
    overall_missing_pct = (missing_cells / total_cells) * 100 if total_cells else 0.0

    # 6) Apply filtering strategy (+ optional min_observed_years)
    dropped_by_rule = pd.DataFrame(columns=[country_col, code_col])

    # Compute non-missing years per country (for optional extra rule)
    non_missing_years = df[year_cols].notna().sum(axis=1)

    # Sparse rule
    if min_non_missing_years is not None:
        keep_mask = non_missing_years >= min_non_missing_years
        dropped_by_rule = df.loc[~keep_mask, [country_col, code_col]].copy()
        df = df.loc[keep_mask].copy()

    # Annual rule
    elif country_missing_threshold is not None:
        country_missing_pct = df[year_cols].isna().mean(axis=1) * 100
        keep_mask = country_missing_pct <= country_missing_threshold

        # Optional: also require minimum observed years
        if min_observed_years is not None:
            keep_mask = keep_mask & (non_missing_years >= min_observed_years)

        dropped_by_rule = df.loc[~keep_mask, [country_col, code_col]].copy()
        df = df.loc[keep_mask].copy()

    # 7) Convert to long format
    long_df = df.melt(
        id_vars=[country_col, code_col],
        value_vars=year_cols,
        var_name="year",
        value_name=value_name
    ).rename(columns={country_col: "country", code_col: "country_code"}).copy()

    long_df["year"] = pd.to_numeric(long_df["year"], errors="coerce").astype("Int64")

    # 8) Missingness flag (pre-imputation)
    if add_missing_flag:
        long_df[f"{value_name}_was_missing"] = long_df[value_name].isna().astype("Int64")

    # 9) Interpolate conservatively
    if interpolate and len(long_df) > 0:
        long_df = long_df.sort_values(["country_code", "year"]).copy()

        def _interp_series(s: pd.Series) -> pd.Series:
            # fill only internal gaps; limit small gaps only
            return s.interpolate(
                method="linear",
                limit_area=interpolate_area,  # "inside" recommended
                limit=max_gap_years
            )

        long_df[value_name] = long_df.groupby("country_code")[value_name].transform(_interp_series)

        # Optional conservative edge fill (limited)
        if edge_fill:
            long_df[value_name] = long_df.groupby("country_code")[value_name].transform(
                lambda s: s.ffill(limit=edge_fill_limit).bfill(limit=edge_fill_limit)
            )

    remaining_missing_pct = float(long_df[value_name].isna().mean() * 100) if len(long_df) > 0 else 0.0

    report = {
        "year_filter_applied_from": min_year,
        "overall_missing_pct_after_year_filter": float(overall_missing_pct),
        "dropped_all_null_countries_count": int(len(dropped_all_null)),
        "dropped_by_rule_countries_count": int(len(dropped_by_rule)),
        "filtering_rule_used": (
            f"min_non_missing_years >= {min_non_missing_years}" if min_non_missing_years is not None
            else (
                f"country_missing_threshold <= {country_missing_threshold}%"
                + (f" AND min_observed_years >= {min_observed_years}" if min_observed_years is not None else "")
            )
        ),
        "imputation": {
            "interpolate": interpolate,
            "interpolate_area": interpolate_area,
            "max_gap_years": max_gap_years,
            "edge_fill": edge_fill,
            "edge_fill_limit": edge_fill_limit,
            "missing_flag_added": add_missing_flag
        },
        "remaining_missing_pct_after_processing": remaining_missing_pct,
        "remaining_rows": int(len(long_df)),
        "remaining_countries": int(long_df["country_code"].nunique()) if len(long_df) > 0 else 0
    }

    return long_df.reset_index(drop=True), report


# ----------------------------
# APPLY V2 RULES TO ALL DATAFRAMES
# ----------------------------

# Annual indicators:
# - 30% missing threshold
# - require enough observed years
# - interpolate only inside, only small gaps (<=2)
# - no edge fill by default (safer)
annual_vars = {
    "gdp": df_gdp,
    "inflation": df_inflation,
    "unemp": df_unemp,
    "labor": df_labor,
    "governance": df_governance,
    "tertiaryeduc": df_tertiaryeduc,
    
}

# Sparse indicators:
# - require at least 3 observed years
# - interpolate inside, allow slightly larger gaps if you want (example: 5)
sparse_vars = {
    "gini": df_gini,
    "poverty": df_poverty,
    "literacy": df_literacy,
    "homicides": df_homicides
}

cleaned_dfs_v2 = {}
reports_v2 = {}

# Annual cleaning (recommended defaults)
for name, df in annual_vars.items():
    cleaned_dfs_v2[name], reports_v2[name] = clean_wdi_like_df_v2(
        df,
        min_year=2000,
        value_name="score",
        country_missing_threshold=30,
        min_non_missing_years=None,
        min_observed_years=15,      # recommended additional coverage rule
        interpolate=True,
        interpolate_area="inside",
        max_gap_years=2,
        edge_fill=False,            # keep False unless you truly need complete edges
        add_missing_flag=True
    )

# Sparse cleaning (recommended defaults)
for name, df in sparse_vars.items():
    cleaned_dfs_v2[name], reports_v2[name] = clean_wdi_like_df_v2(
        df,
        min_year=2000,
        value_name="score",
        country_missing_threshold=None,
        min_non_missing_years=3,
        interpolate=True,
        interpolate_area="inside",
        max_gap_years=5,            # sparse can tolerate slightly larger internal gaps if needed
        edge_fill=False,
        add_missing_flag=True
    )

# Summary of remaining countries per dataset
countries_summary_v2 = {k: v["remaining_countries"] for k, v in reports_v2.items()}
countries_summary_v2

{'gdp': 240,
 'inflation': 224,
 'unemp': 235,
 'labor': 235,
 'governance': 198,
 'tertiaryeduc': 140,
 'gini': 140,
 'poverty': 155,
 'literacy': 150,
 'homicides': 227}

We cleaned and standardized all World Bank–style datasets using a consistent pipeline designed for cross-country panel analysis from 2000 onward. First, we restricted every dataset to years ≥2000 to align time coverage and reduce structural missingness from earlier decades. We converted all year columns to numeric, then removed countries with no available observations in the selected period (100% missing), since they cannot be imputed meaningfully. After that, we applied two different inclusion rules depending on the indicator type: for annual macroeconomic indicators (GDP, inflation, unemployment, labor, governance, and tertiary education), we dropped countries with more than 30% missing values and then used within-country linear interpolation to fill small gaps and produce a complete year-by-year panel. For sparse survey/census indicators (Gini, poverty, and literacy), we avoided percent-missing thresholds (which would remove most countries) and instead retained countries with at least three observed data points, then interpolated only within the observed range to avoid extrapolating values beyond the first and last measurement. Finally, each dataset was reshaped from wide format into a standardized long format with columns country, country_code, year, and score, producing harmonized panel-ready datasets for merging and modeling.

# Integration of Harmonized World Bank Indicators into a Unified Panel Dataset

In [4]:
import pandas as pd
from functools import reduce

# 1) Usar cleaned_dfs_v2
ready = {}

for indicator, df in cleaned_dfs_v2.items():
    tmp = df.copy()

    # Detectar columnas dinámicamente
    value_col = "score"
    flag_col = f"{value_col}_was_missing"

    # Renombrar columnas
    tmp = tmp[["country_code", "year", "country", value_col, flag_col]]

    tmp = tmp.rename(columns={
        value_col: indicator,
        flag_col: f"{indicator}_was_missing"
    })

    ready[indicator] = tmp


# 2) Country lookup (igual que antes)
country_lookup = (
    pd.concat([d[["country_code", "country"]] for d in ready.values()], ignore_index=True)
      .dropna()
      .drop_duplicates(subset=["country_code"])
)


# 3) Solo country_code + year + columnas del indicador
dfs_to_merge = []
for indicator, df in ready.items():
    dfs_to_merge.append(
        df[["country_code", "year", indicator, f"{indicator}_was_missing"]]
    )


# 4) Merge secuencial
merged = reduce(
    lambda left, right: pd.merge(left, right, on=["country_code", "year"], how="outer"),
    dfs_to_merge
)


# 5) Agregar country
merged = merged.merge(country_lookup, on="country_code", how="left")


# 6) Ordenar columnas
indicator_cols = list(ready.keys())

value_cols = indicator_cols
flag_cols = [f"{col}_was_missing" for col in indicator_cols]

merged = merged[
    ["country_code", "country", "year"] + value_cols + flag_cols
].sort_values(["country_code", "year"]).reset_index(drop=True)

merged.head()

,country_code,country,year,gdp,inflation,unemp,labor,governance,tertiaryeduc,gini,...,gdp_was_missing,inflation_was_missing,unemp_was_missing,labor_was_missing,governance_was_missing,tertiaryeduc_was_missing,gini_was_missing,poverty_was_missing,literacy_was_missing,homicides_was_missing
0,ABW,Aruba,2000,30245.706974,4.044021,NaN,NaN,NaN,NaN,NaN,...,0,0,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,1
1,ABW,Aruba,2001,31920.239073,2.883604,NaN,NaN,NaN,NaN,NaN,...,0,0,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,0
2,ABW,Aruba,2002,31888.508737,3.315247,NaN,NaN,NaN,NaN,NaN,...,0,0,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,0
3,ABW,Aruba,2003,32507.084315,3.656365,NaN,NaN,NaN,NaN,NaN,...,0,0,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,0
4,ABW,Aruba,2004,35059.273098,2.529129,NaN,NaN,1.281389,NaN,NaN,...,0,0,<NA>,<NA>,0,<NA>,<NA>,<NA>,<NA>,0


In [5]:
merged['country'].nunique()

260

We took all the cleaned indicator dataframes stored in cleaned_dfs (each one already standardized to country, country_code, year, and score) and prepared them to be combined into a single longitudinal panel dataset. For each indicator, we selected only the key columns (country_code, year, country, score) and renamed the score column to the indicator’s name so that each dataset would contribute one uniquely named feature column. To avoid inconsistencies in country naming across sources, we built a separate country_lookup table that maps each country_code to a single country name, and then we performed the merges using only the stable keys country_code and year. Next, we merged all indicators sequentially with an outer join, which preserves the largest possible country–year panel and keeps missing values as NaN when an indicator is not available for a given country-year. Finally, we merged back the country name from the lookup table and reordered the columns into the final structure (country_code, country, year, followed by one column per indicator), sorted by country_code and year so the dataset is organized and ready for analysis or modeling.

In [6]:
import pandas as pd
from functools import reduce

# 1) Preparar cada df (solo valor principal)
ready = {}

for indicator, df in cleaned_dfs_v2.items():
    tmp = df.copy()

    tmp = tmp[["country_code", "year", "country", "score"]]

    tmp = tmp.rename(columns={"score": indicator})

    ready[indicator] = tmp


# 2) Country lookup
country_lookup = (
    pd.concat([d[["country_code", "country"]] for d in ready.values()], ignore_index=True)
      .dropna()
      .drop_duplicates(subset=["country_code"])
)


# 3) Lista para merge (solo country_code + year + indicador)
dfs_to_merge = []
for indicator, df in ready.items():
    dfs_to_merge.append(
        df[["country_code", "year", indicator]]
    )


# 4) Merge secuencial
merged = reduce(
    lambda left, right: pd.merge(left, right, on=["country_code", "year"], how="outer"),
    dfs_to_merge
)


# 5) Agregar country
merged = merged.merge(country_lookup, on="country_code", how="left")


# 6) Ordenar columnas
indicator_cols = list(ready.keys())

merged = merged[
    ["country_code", "country", "year"] + indicator_cols
].sort_values(["country_code", "year"]).reset_index(drop=True)

merged.head()

,country_code,country,year,gdp,inflation,unemp,labor,governance,tertiaryeduc,gini,poverty,literacy,homicides
0,ABW,Aruba,2000,30245.706974,4.044021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,Aruba,2001,31920.239073,2.883604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.429434
2,ABW,Aruba,2002,31888.508737,3.315247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.471747
3,ABW,Aruba,2003,32507.084315,3.656365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.333225
4,ABW,Aruba,2004,35059.273098,2.529129,NaN,NaN,1.281389,NaN,NaN,NaN,NaN,2.145635


In [7]:
merged['country'].nunique()

260

In [8]:
print("Países únicos:", merged["country_code"].nunique())
print("Años únicos:", merged["year"].nunique())
print("Shape:", merged.shape)
print("Missing % por columna:")
print(merged.isna().mean() * 100)

Países únicos: 260
Años únicos: 26
Shape: (6760, 13)
Missing % por columna:
country_code     0.000000
country          0.000000
year             0.000000
gdp             11.420118
inflation       18.387574
unemp            9.822485
labor            9.822485
governance      30.680473
tertiaryeduc    50.399408
gini            60.502959
poverty         55.147929
literacy        58.920118
homicides       43.550296
dtype: float64


In [9]:
merged.to_parquet("cleaned_data/df_merged.parquet", index=False)